In [1]:
!pip install transformers peft bitsandbytes accelerate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 12.3 MB/s eta 0:00:00


In [3]:
import transformers
import peft
import bitsandbytes
import accelerate

print("transformers:", transformers.__version__)
print("peft:", peft.__version__)
print("bitsandbytes:", bitsandbytes.__version__)
print("accelerate:", accelerate.__version__)

transformers: 5.0.0
peft: 0.18.1
bitsandbytes: 0.49.2
accelerate: 1.12.0


In [4]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

print("Base model loaded in FP16!")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.20G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Base model loaded in FP16!


In [5]:
from peft import PeftModel

model_with_adapters = PeftModel.from_pretrained(
    base_model,
    "/content"
)

print("Adapters loaded successfully!")

Adapters loaded successfully!


In [6]:
merged_model = model_with_adapters.merge_and_unload()
print("Model merged successfully!")

Model merged successfully!


In [7]:
import os
os.makedirs("/content/merged_model", exist_ok=True)

merged_model.save_pretrained("/content/merged_model")
tokenizer.save_pretrained("/content/merged_model")

print("Merged model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved!


In [8]:
from transformers import BitsAndBytesConfig

bnb_int8 = BitsAndBytesConfig(load_in_8bit=True)

model_int8 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_int8,
    device_map="auto"
)

print("INT8 model loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT8 model loaded!


In [9]:
os.makedirs("/content/quantized/model-int8", exist_ok=True)

model_int8.save_pretrained("/content/quantized/model-int8")
tokenizer.save_pretrained("/content/quantized/model-int8")

print("INT8 model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT8 model saved!


In [10]:
from transformers import BitsAndBytesConfig

bnb_int4 = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)

model_int4 = AutoModelForCausalLM.from_pretrained(
    "/content/merged_model",
    quantization_config=bnb_int4,
    device_map="auto"
)

print("INT4 model loaded!")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

INT4 model loaded!


In [11]:
os.makedirs("/content/quantized/model-int4", exist_ok=True)

model_int4.save_pretrained("/content/quantized/model-int4")
tokenizer.save_pretrained("/content/quantized/model-int4")

print("INT4 model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

INT4 model saved!


In [12]:
!git clone https://github.com/ggerganov/llama.cpp.git
!pip install -r llama.cpp/requirements.txt -q
print("llama.cpp ready!")

Cloning into 'llama.cpp'...
remote: Enumerating objects: 80923, done.
remote: Counting objects: 100% (163/163), done.
remote: Compressing objects: 100% (103/103), done.
remote: Total 80923 (delta 108), reused 60 (delta 60), pack-reused 80760 (from 2)
Receiving objects: 100% (80923/80923), 306.36 MiB | 34.05 MiB/s, done.
Resolving deltas: 100% (58440/58440), done.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 370.2 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 78.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.7/12.7 MB 87.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 32.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 114.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 96.2/96.2 kB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [14]:
!python /content/llama.cpp/convert_hf_to_gguf.py /content/merged_model \
    --outfile /content/quantized/model.gguf \
    --outtype q8_0

print("GGUF conversion done!")

INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: LlamaForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:output.weight,               torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:token_embd.weight,           torch.float16 --> Q8_0, shape = {2048, 32000}
INFO:hf-to-gguf:blk.0.attn_norm.weight,      torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.ffn_down.weight,       torch.float16 --> Q8_0, shape = {5632, 2048}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,       torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_up.weight,         torch.float16 --> Q8_0, shape = {2048, 5632}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,       torch.float16 --> F32, shape = {2048}
INFO:hf-to-gguf:blk.0.attn_k.weight,         torch.float16 --> Q8_0, shape = {2048, 256}
INFO:hf-to-gguf:blk.0.attn_out

In [13]:
from huggingface_hub import hf_hub_download
import shutil, os

tokenizer_model = hf_hub_download(
    repo_id="TinyLlama/TinyLlama-1.1B-Chat-v1.0",
    filename="tokenizer.model"
)

shutil.copy(tokenizer_model, "/content/merged_model/tokenizer.model")

print(os.listdir("/content/merged_model"))

['config.json', 'tokenizer_config.json', 'chat_template.jinja', 'generation_config.json', 'tokenizer.json', 'tokenizer.model', 'model.safetensors']


In [15]:
import os

def folder_size(path):
    total = sum(os.path.getsize(os.path.join(r, f))
                for r, _, files in os.walk(path)
                for f in files)
    return round(total / (1024 * 1024 * 1024), 2)

fp16_size = folder_size("/content/merged_model")
int8_size = folder_size("/content/quantized/model-int8")
int4_size = folder_size("/content/quantized/model-int4")
gguf_size = round(os.path.getsize("/content/quantized/model.gguf") / (1024 * 1024 * 1024), 2)

print(f"FP16 size: {fp16_size} GB")
print(f"INT8 size: {int8_size} GB")
print(f"INT4 size: {int4_size} GB")
print(f"GGUF size: {gguf_size} GB")

FP16 size: 2.05 GB
INT8 size: 1.15 GB
INT4 size: 0.76 GB
GGUF size: 1.09 GB


In [16]:
!pip install llama-cpp-python -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.7/50.7 MB 18.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 3.0 MB/s eta 0:00:00


In [17]:
from llama_cpp import Llama

gguf_model = Llama(
    model_path="/content/quantized/model.gguf",
    n_gpu_layers=-1,
    verbose=False
)

print("GGUF model loaded!")

llama_context: n_ctx_per_seq (512) < n_ctx_train (2048) -- the full capacity of the model will not be utilized


GGUF model loaded!


In [18]:
import time
import torch

def measure_inference(model, tokenizer, name, prompt="Write a Python function to add two numbers"):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    # Reset VRAM stats
    torch.cuda.reset_peak_memory_stats()

    # Measure latency and speed
    start = time.time()
    outputs = model.generate(
        **inputs,
        max_new_tokens=100,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )
    end = time.time()

    # Calculate metrics
    tokens = outputs.shape[1]
    latency = round(end - start, 2)
    speed = round(tokens / latency, 2)
    vram = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
    text = tokenizer.decode(outputs[0], skip_special_tokens=True)

    print(f"\n {name}")
    print(f"Tokens/sec : {speed}")
    print(f"VRAM       : {vram} GB")
    print(f"Latency    : {latency} sec")
    print(f"Output     : {text}")

    return speed, vram, latency, text

# Measure all models
base_speed, base_vram, base_latency, base_output = measure_inference(base_model, tokenizer, "Base Model")
ft_speed, ft_vram, ft_latency, ft_output = measure_inference(merged_model, tokenizer, "Fine-tuned Model")


=== Base Model ===
Tokens/sec : 10.51
VRAM       : 4.0 GB
Latency    : 3.71 sec
Output     : Write a Python function to add two numbers.

# Function to add two numbers 
def sum_of_two(x, y):  
    return x + y

=== Fine-tuned Model ===
Tokens/sec : 20.31
VRAM       : 4.0 GB
Latency    : 1.92 sec
Output     : Write a Python function to add two numbers.

# Function to add two numbers 
def sum_of_two(x, y):  
    return x + y


In [21]:
start = time.time()
output = gguf_model.create_chat_completion(
    messages=[{"role": "user", "content": "Write a Python function to add two numbers"}],
    max_tokens=150
)
end = time.time()

tokens = output['usage']['completion_tokens']
gguf_speed = round(tokens / (end - start), 2)
gguf_latency = round(end - start, 2)
gguf_output = output['choices'][0]['message']['content']

print(f"\n GGUF Model ")
print(f"Tokens/sec : {gguf_speed}")
print(f"VRAM       : N/A (llama.cpp manages memory)")
print(f"Latency    : {gguf_latency} sec")
print(f"Output     : {gguf_output}")


 GGUF Model 
Tokens/sec : 7.33
VRAM       : N/A (llama.cpp manages memory)
Latency    : 12.55 sec
Output     : def add_numbers(num1, num2):
    # Add the two numbers
    return num1 + num2

# Example usage
print(add_numbers(5, 10)) # Output: 15

# Example usage with error handling
try:
    print(add_numbers(5, 10))
except ValueError:
    print("Error: Both numbers must be integers")


In [23]:
from transformers import TextStreamer

def stream_transformer(model, tokenizer, name, prompt):
    print(f"\n Streaming: {name} ")
    streamer = TextStreamer(tokenizer, skip_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    model.generate(
        **inputs,
        max_new_tokens=100,
        streamer=streamer,
        eos_token_id=tokenizer.eos_token_id,
        repetition_penalty=1.2
    )

def stream_gguf(model, name, prompt):
    print(f"\n Streaming: {name} ")
    output = model.create_chat_completion(
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
        stream=True
    )
    for chunk in output:
        delta = chunk['choices'][0]['delta']
        if 'content' in delta:
            print(delta['content'], end='', flush=True)
    print()

prompt = "Write a Python function to add two numbers"

# Stream all 3 models
stream_transformer(base_model, tokenizer, "Base Model", prompt)
stream_transformer(merged_model, tokenizer, "Fine-tuned Model", prompt)
stream_gguf(gguf_model, "GGUF Model", prompt)


 Streaming: Base Model 
.

# Function to add two numbers 
def sum_of_two(x, y):  
   return x + y</s>

 Streaming: Fine-tuned Model 
.

# Function to add two numbers 
def sum_of_two(x, y):  
   return x + y</s>

 Streaming: GGUF Model 
def add_numbers(num1, num2):
    """
    This function takes two numbers as input and adds them together.
    """
    return num1 + num2

# Example usage
print(add_numbers(10, 20)) # Output: 30


In [29]:
print(" Batch Inference ")

transformer_prompts = [
    "def add_numbers(a, b):\n    return a + b\n\nprint(add_numbers(10, 20))",
    "def multiply_numbers(a, b):\n    return a * b\n\nprint(multiply_numbers(2, 3))",
]

gguf_prompts = [
    "Write only Python code:\ndef add_numbers(a, b):",
    "Write only Python code:\ndef multiply_numbers(a, b):",
]

def batch_transformer(model, tokenizer, name, prompts):
    print(f"\n Batch: {name} ")
    for i, prompt in enumerate(prompts):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2,
            temperature=0,
            do_sample=False
        )
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\nPrompt {i+1}:")
        print(f"Output : {text}")

def batch_gguf(model, name, prompts):
    print(f"\n Batch: {name}")
    for i, prompt in enumerate(prompts):
        output = model.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=100,
            temperature=0
        )
        text = output['choices'][0]['message']['content']
        print(f"\nPrompt {i+1}:")
        print(f"Output : {text}")

batch_transformer(base_model, tokenizer, "Base Model", transformer_prompts)
batch_transformer(merged_model, tokenizer, "Fine-tuned Model", transformer_prompts)
batch_gguf(gguf_model, "GGUF Model", gguf_prompts)

The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


 Batch Inference 

 Batch: Base Model 

Prompt 1:
Output : def add_numbers(a, b):
    return a + b

print(add_numbers(10, 20)) # Output: 30

Prompt 2:
Output : def multiply_numbers(a, b):
    return a * b

print(multiply_numbers(2, 3)) # Output: 6

 Batch: Fine-tuned Model 

Prompt 1:
Output : def add_numbers(a, b):
    return a + b

print(add_numbers(10, 20)) # Output: 30

Prompt 2:
Output : def multiply_numbers(a, b):
    return a * b

print(multiply_numbers(2, 3)) # Output: 6

 Batch: GGUF Model

Prompt 1:
Output : def add_numbers(a, b):
    # Add the two numbers
    return a + b

# Example usage
print(add_numbers(10, 20)) # Output: 30

Prompt 2:
Output : def multiply_numbers(a, b):
    # Calculate the product
    product = a * b

    # Print the result
    print("The product of {} and {} is {}".format(a, b, product))


In [31]:
print("Multi-prompt Test")

multi_prompts = [
    "Write a Python function to reverse a string",
    "Write a Python function to find maximum in a list",
    "Write only Python code:\ndef factorial(n):"
]

def multi_prompt_transformer(model, tokenizer, name, prompts):
    print(f"\nMulti-prompt: {name}")
    for i, prompt in enumerate(prompts):
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            eos_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.2
        )
        text = tokenizer.decode(outputs[0], skip_special_tokens=True)
        print(f"\nPrompt {i+1}: {prompt}")
        print(f"Output : {text}")

def multi_prompt_gguf(model, name, prompts):
    print(f"\nMulti-prompt: {name}")
    for i, prompt in enumerate(prompts):
        output = model.create_chat_completion(
            messages=[{"role": "user", "content": prompt}],
            max_tokens=150,
            temperature=0
        )
        text = output['choices'][0]['message']['content']
        print(f"\nPrompt {i+1}: {prompt}")
        print(f"Output : {text}")

multi_prompt_transformer(base_model, tokenizer, "Base Model", multi_prompts)
multi_prompt_transformer(merged_model, tokenizer, "Fine-tuned Model", multi_prompts)
multi_prompt_gguf(gguf_model, "GGUF Model", multi_prompts)

Multi-prompt Test

Multi-prompt: Base Model

Prompt 1: Write a Python function to reverse a string
Output : Write a Python function to reverse a string.

# Function to Reverse String 
def reverse_string(s):  
    # Create an empty list to store the reversed string  
    r = []  
    
    # Loop through each character in the string and add it to the list  
    for i in range (len(s)):  
        r.append(s[i])  
        
    return ''.join(r)  

# Example usage of the function  
print(reverse_string("Hello World"))

Prompt 2: Write a Python function to find maximum in a list
Output : Write a Python function to find maximum in a list.

# Function to find the max element of a given list 
def find_max(lst):  
    # Initialize variable for storing max value 
    max = lst[0] 
    
    # Loop through all elements of the list and update max if it is greater than current max 
    for i in range (1, len(lst)): 
        if lst[i] > max: 
            max = lst[i] 
            
    return max

Prom

In [33]:
import csv

results = [
    ["Model", "Tokens/sec", "VRAM", "Latency", "Accuracy"],
    ["Base", base_speed, base_vram, base_latency, "Good"],
    ["Fine-tuned", ft_speed, ft_vram, ft_latency, "Best"],
    ["GGUF", gguf_speed, "N/A", gguf_latency, "Mixed"]
]

with open("/content/results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(results)

print("results.csv saved!")

results.csv saved!
